<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/term_project_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import warnings
warnings.filterwarnings('ignore')

In [57]:
path = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/titanic/train.csv"

import pandas as pd
df = pd.read_csv(path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [58]:
df = df.drop(columns=['PassengerId', 'Ticket'])
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C85,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,NaN,S


In [59]:
import pandas as pd

# Name 컬럼에서 호칭(Title) 추출
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['Title'].value_counts()

,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Dr,7
Rev,6
Col,2
Mlle,2
Major,2


In [60]:
# 호칭 단순화 매핑 사전 정의
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mme': 'Mrs',
    # 희귀 호칭 묶기
    'Lady': 'Rare',
    'Countess': 'Rare',
    'Capt': 'Rare',
    'Col': 'Rare',
    'Don': 'Rare',
    'Dr': 'Rare',
    'Major': 'Rare',
    'Rev': 'Rare',
    'Sir': 'Rare',
    'Jonkheer': 'Rare',
    'Dona': 'Rare'
}

df['Title'] = df['Title'].map(title_mapping)
df['Title'].value_counts()

,count
Title,
Mr,517
Miss,185
Mrs,126
Master,40
Rare,23


In [61]:
df = df.drop(columns=['Name'])
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,Title
0,0,3,male,22.0,1,0,7.2500,NaN,S,Mr
1,1,1,female,38.0,1,0,71.2833,C85,C,Mrs
2,1,3,female,26.0,0,0,7.9250,NaN,S,Miss
3,1,1,female,35.0,1,0,53.1000,C123,S,Mrs
4,0,3,male,35.0,0,0,8.0500,NaN,S,Mr
...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,NaN,S,Rare
887,1,1,female,19.0,0,0,30.0000,B42,S,Miss
888,0,3,female,NaN,1,2,23.4500,NaN,S,Miss
889,1,1,male,26.0,0,0,30.0000,C148,C,Mr


In [62]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    # 데이터프레임의 각 컬럼별 데이터 타입, 결측치 수/비율, 대표값 5개를 요약한 DataFrame을 반환합니다.
    summary_data = []
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        null_cnt = df[col].isnull().sum()
        null_ratio = round((null_cnt / len(df)) * 100, 2)
        unique_vals = df[col].dropna().unique()
        sample_vals = list(unique_vals[:7])

        summary_data.append({
            'Column': col,
            'N_Unique': n_unique,
            'Dtype': dtype,
            'Null_Count': null_cnt,
            'Null_Ratio(%)': null_ratio,
            'Sample_Values_5': sample_vals
        })

    return pd.DataFrame(summary_data)

In [63]:
summarize(df)

,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,Survived,2,int64,0,0.00,"[0, 1]"
1,Pclass,3,int64,0,0.00,"[3, 1, 2]"
2,Sex,2,object,0,0.00,"[male, female]"
3,Age,88,float64,177,19.87,"[22.0, 38.0, 26.0, 35.0, 54.0, 2.0, 27.0]"
4,SibSp,7,int64,0,0.00,"[1, 0, 3, 4, 2, 5, 8]"
5,Parch,7,int64,0,0.00,"[0, 1, 2, 5, 3, 4, 6]"
6,Fare,248,float64,0,0.00,"[7.25, 71.2833, 7.925, 53.1, 8.05, 8.4583, 51...."
7,Cabin,147,object,687,77.10,"[C85, C123, E46, G6, C103, D56, A6]"
8,Embarked,3,object,2,0.22,"[S, C, Q]"
9,Title,5,object,0,0.00,"[Mr, Mrs, Miss, Master, Rare]"


In [64]:
df.groupby('Embarked')['Survived'].mean()

,Survived
Embarked,
C,0.553571
Q,0.389610
S,0.336957


In [65]:
df = df.drop(columns=['Cabin'])
summarize(df)

,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,Survived,2,int64,0,0.00,"[0, 1]"
1,Pclass,3,int64,0,0.00,"[3, 1, 2]"
2,Sex,2,object,0,0.00,"[male, female]"
3,Age,88,float64,177,19.87,"[22.0, 38.0, 26.0, 35.0, 54.0, 2.0, 27.0]"
4,SibSp,7,int64,0,0.00,"[1, 0, 3, 4, 2, 5, 8]"
5,Parch,7,int64,0,0.00,"[0, 1, 2, 5, 3, 4, 6]"
6,Fare,248,float64,0,0.00,"[7.25, 71.2833, 7.925, 53.1, 8.05, 8.4583, 51...."
7,Embarked,3,object,2,0.22,"[S, C, Q]"
8,Title,5,object,0,0.00,"[Mr, Mrs, Miss, Master, Rare]"


In [66]:
df['Embarked'].value_counts()

,count
Embarked,
S,644
C,168
Q,77


In [67]:
df['Age'].value_counts()

,count
Age,
24.00,30
22.00,27
18.00,26
28.00,25
30.00,25
...,...
24.50,1
0.67,1
0.42,1


In [68]:
# Pclass와 Title 조합별 Age 중앙값으로 대체
df['Age'] = df['Age'].fillna(df.groupby(['Pclass', 'Title'])['Age'].transform('median'))

In [69]:
# 최빈값('S') 구해서 결측치 채우기
mode_embarked = df['Embarked'].mode()[0]  # 'S'
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

In [70]:
summarize(df)

,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,Survived,2,int64,0,0.0,"[0, 1]"
1,Pclass,3,int64,0,0.0,"[3, 1, 2]"
2,Sex,2,object,0,0.0,"[male, female]"
3,Age,89,float64,0,0.0,"[22.0, 38.0, 26.0, 35.0, 54.0, 2.0, 27.0]"
4,SibSp,7,int64,0,0.0,"[1, 0, 3, 4, 2, 5, 8]"
5,Parch,7,int64,0,0.0,"[0, 1, 2, 5, 3, 4, 6]"
6,Fare,248,float64,0,0.0,"[7.25, 71.2833, 7.925, 53.1, 8.05, 8.4583, 51...."
7,Embarked,3,object,0,0.0,"[S, C, Q]"
8,Title,5,object,0,0.0,"[Mr, Mrs, Miss, Master, Rare]"


In [71]:
# 1. 모델 및 평가 지표 라이브러리
import numpy as np
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    average_precision_score,  # PR_AUC
    roc_auc_score,             # AUROC
)

# -------------------------------------------------------------------
# [전처리 과정]
# 이미 결측치가 모두 처리된 데이터프레임 `df`가 존재한다고 가정합니다.
# -------------------------------------------------------------------

# 1) 범주형 변수 인코딩 (Sex, Embarked, Title)
df_encoded = df.copy()

# Sex: binary mapping
df_encoded['Sex'] = df_encoded['Sex'].map({'male': 0, 'female': 1})

# Embarked, Title: One-Hot Encoding
df_encoded = pd.get_dummies(df_encoded, columns=['Embarked', 'Title'], drop_first=True)

# 2) Feature(X)와 Target(y) 분리
X = df_encoded.drop('Survived', axis=1)
y = df_encoded['Survived']

# 3) 수치형 데이터 스케일링 (KNN, Logistic Regression 등에 필수)
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# -------------------------------------------------------------------
# [모델 정의 및 Cross Validation 평가]
# -------------------------------------------------------------------

# 평가할 모델 목록 정의
models = {
    'KNN': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

# Stratified K-Fold 설정 (5-Fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_list = []

for model_name, model in models.items():

    # 지표별 Fold 결과 저장용 리스트
    fpr_list = []
    err_list = []
    recall_list = []
    precision_list = []
    f1_list = []
    pr_auc_list = []
    auroc_list = []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # 모델 학습 및 예측
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]  # 확률 값 (AUC 계산용)

        # 오탐률(FPR) 계산: FP / (FP + TN)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        errors = fn+fp

        # 지표 저장
        fpr_list.append(fpr)
        err_list.append(errors)
        recall_list.append(recall_score(y_test, y_pred, zero_division=0))
        precision_list.append(precision_score(y_test, y_pred, zero_division=0))
        f1_list.append(f1_score(y_test, y_pred, zero_division=0))
        pr_auc_list.append(average_precision_score(y_test, y_prob))
        auroc_list.append(roc_auc_score(y_test, y_prob))

    # 평균 결과 수집
    results_list.append({
        'Model': model_name,
        'FN+FP (오분류 수)' : np.mean(err_list),
        'FPR (오탐률)': np.mean(fpr_list),
        'Recall (재현율)': np.mean(recall_list),
        'Precision (정밀도)': np.mean(precision_list),
        'F1-Score': np.mean(f1_list),
        'PR_AUC': np.mean(pr_auc_list),
        'AUROC': np.mean(auroc_list)
    })

# 결과 데이터프레임 생성 및 출력
results_df = pd.DataFrame(results_list)
results_df = results_df.round(4) # 소수점 4자리 정리

results_df

,Model,FN+FP (오분류 수),FPR (오탐률),Recall (재현율),Precision (정밀도),F1-Score,PR_AUC,AUROC
0,KNN,32.4,0.1294,0.7339,0.7816,0.7561,0.7827,0.8558
1,Logistic Regression,30.6,0.1221,0.7483,0.7932,0.7694,0.8541,0.8722
2,Naive Bayes,38.6,0.1585,0.6898,0.7309,0.7087,0.7654,0.8375
3,Decision Tree,36.0,0.1603,0.7310,0.7397,0.7349,0.6436,0.7823
4,Random Forest,30.6,0.1184,0.7427,0.7964,0.7685,0.8226,0.8730
5,XGBoost,31.8,0.1330,0.7484,0.7783,0.7630,0.8484,0.8743


In [72]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# 1. 탐색할 하이퍼파라미터 분포(Grid) 정의
param_distributions = {
    'n_estimators': [50, 100, 150, 200, 300],          # 트리 개수
    'max_depth': [3, 4, 5, 6, 7, 8],                   # 트리의 최대 깊이
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],     # 학습률
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],            # 샘플링 비율
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],     # 피처 탐색 비율
    'min_child_weight': [1, 2, 3, 5, 7],               # 노드 분할 최소 가중치 합
    'gamma': [0, 0.1, 0.2, 0.3, 0.5]                   # 리프 노드 추가 분할 소요 최소 손실 감소량
}

# 2. 기본 XGBoost 모델 및 K-Fold 설정
base_xgb = XGBClassifier(random_state=42, eval_metric='logloss')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. RandomizedSearchCV 설정
# n_iter=50: 조합 중 50개를 랜덤하게 추출하여 평가
random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='f1',               # 튜닝 기준 지표 (f1, roc_auc 등으로 변경 가능)
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 4. 하이퍼파라미터 탐색 수행
random_search.fit(X, y)

print("=" * 50)
print(f"최적 하이퍼파라미터: {random_search.best_params_}")
print(f"최고 CV F1-Score: {random_search.best_score_:.4f}")
print("=" * 50)

# 5. 최적 모델(Best Estimator)을 사용하여 Cross Validation 성능 재평가
best_xgb = random_search.best_estimator_

fn_fp_list = []
fpr_list = []
recall_list = []
precision_list = []
f1_list = []
pr_auc_list = []
auroc_list = []

for train_idx, test_idx in tqdm(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    best_xgb.fit(X_train, y_train)
    y_pred = best_xgb.predict(X_test)
    y_prob = best_xgb.predict_proba(X_test)[:, 1]

    # 혼동 행렬 지표 계산
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    fn_fp_list.append(fn + fp)
    fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    recall_list.append(recall_score(y_test, y_pred, zero_division=0))
    precision_list.append(precision_score(y_test, y_pred, zero_division=0))
    f1_list.append(f1_score(y_test, y_pred, zero_division=0))
    pr_auc_list.append(average_precision_score(y_test, y_prob))
    auroc_list.append(roc_auc_score(y_test, y_prob))

# 6. 결과 출력
tuned_result = pd.DataFrame([{
    'Model': 'XGBoost (Tuned)',
    'FN+FP (오분류 수)': np.mean(fn_fp_list),
    'FPR (오탐률)': np.mean(fpr_list),
    'Recall (재현율)': np.mean(recall_list),
    'Precision (정밀도)': np.mean(precision_list),
    'F1-Score': np.mean(f1_list),
    'PR_AUC': np.mean(pr_auc_list),
    'AUROC': np.mean(auroc_list)
}]).round(4)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 0.7}
최고 CV F1-Score: 0.7920


5it [00:01,  2.51it/s]


In [73]:
tuned_result

,Model,FN+FP (오분류 수),FPR (오탐률),Recall (재현율),Precision (정밀도),F1-Score,PR_AUC,AUROC
0,XGBoost (Tuned),27.4,0.102,0.7631,0.8235,0.792,0.8612,0.885


In [74]:
results_df.tail(1)

,Model,FN+FP (오분류 수),FPR (오탐률),Recall (재현율),Precision (정밀도),F1-Score,PR_AUC,AUROC
5,XGBoost,31.8,0.133,0.7484,0.7783,0.763,0.8484,0.8743


In [75]:
import joblib

# 1. 최적의 모델(best_estimator_) 저장하기
joblib.dump(best_xgb, 'best_xgb_model.pkl')
print("최적 모델 저장 완료!")

# 2. 필요한 경우 RandomizedSearchCV 객체 전체 저장하기
# joblib.dump(random_search, 'xgb_random_search.pkl')


최적 모델 저장 완료!


In [76]:
path = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/titanic/test.csv"
tf = pd.read_csv(path)
tf

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [77]:
target = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/titanic/gendermodel.csv"
target = pd.read_csv(target)
tf = tf.merge(target, on='PassengerId')
tf

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,1
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,0
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,0
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,1
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S,0
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,1
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,0
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S,0


In [78]:
summarize(tf)

,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,PassengerId,418,int64,0,0.00,"[892, 893, 894, 895, 896, 897, 898]"
1,Pclass,3,int64,0,0.00,"[3, 2, 1]"
2,Name,418,object,0,0.00,"[Kelly, Mr. James, Wilkes, Mrs. James (Ellen N..."
3,Sex,2,object,0,0.00,"[male, female]"
4,Age,79,float64,86,20.57,"[34.5, 47.0, 62.0, 27.0, 22.0, 14.0, 30.0]"
5,SibSp,7,int64,0,0.00,"[0, 1, 2, 3, 4, 5, 8]"
6,Parch,8,int64,0,0.00,"[0, 1, 3, 2, 4, 6, 5]"
7,Ticket,363,object,0,0.00,"[330911, 363272, 240276, 315154, 3101298, 7538..."
8,Fare,169,float64,1,0.24,"[7.8292, 7.0, 9.6875, 8.6625, 12.2875, 9.225, ..."
9,Cabin,76,object,327,78.23,"[B45, E31, B57 B59 B63 B66, B36, A21, C78, D34]"


In [79]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    average_precision_score,  # PR_AUC
    roc_auc_score,             # AUROC
)

# =====================================================================
# 1. 전처리 함수 정의 (Title 추출, 결측치 처리, 원-핫 인코딩)
# =====================================================================
def preprocess_data(df_raw, is_test=True, train_title_medians=None, train_fare_median=None):
    df = df_raw.copy()

    # 1) 불필요한 컬럼 제거 (PassengerId, Ticket, Cabin)
    df.drop(['PassengerId', 'Ticket', 'Cabin'], axis=1, inplace=True, errors='ignore')

    # 2) Name 컬럼에서 호칭(Title) 추출 및 희귀 호칭 매핑
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    title_mapping = {
        'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
        'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
        'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
    }
    df['Title'] = df['Title'].map(title_mapping).fillna('Rare')
    df.drop('Name', axis=1, inplace=True, errors='ignore')

    # 3) Embarked 결측치 처리 (최빈값 'S')
    df['Embarked'] = df['Embarked'].fillna('S')

    # 4) Age 결측치 처리 (Title별 중앙값 적용)
    if train_title_medians is not None:
        df['Age'] = df['Age'].fillna(df['Title'].map(train_title_medians))
    else:
        df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))
    df['Age'] = df['Age'].fillna(df['Age'].median())  # 예외 대체값

    # 5) Fare 결측치 처리 (중앙값 적용)
    if train_fare_median is not None:
        df['Fare'] = df['Fare'].fillna(train_fare_median)
    else:
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())

    # 6) 범주형 데이터 변환 (Sex 바이너리, Embarked & Title 원-핫 인코딩)
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True)

    # 7) 학습 시 생성된 원-핫 인코딩 컬럼 구조에 맞추기 (없을 경우 0으로 자동 채움)
    expected_cols = [
        'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
        'Embarked_Q', 'Embarked_S',
        'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare'
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = 0

    # Target(Survived)을 제외한 피처 순서 정렬
    feature_cols = [col for col in expected_cols if col in df.columns]

    return df, feature_cols


# =====================================================================
# 2. 데이터 불러오기 및 전처리 수행
# =====================================================================
# 예: 원본 테스트 데이터셋 `test_df`에 정답(`Survived`) 컬럼이 포함되어 있는 경우
df_processed, feature_cols = preprocess_data(tf)

X_test = df_processed[feature_cols]
y_test = df_processed['Survived']


# =====================================================================
# 3. 저장된 피클 모델 로드
# =====================================================================
model_obj = joblib.load('best_xgb_model.pkl')

# RandomizedSearchCV 객체 전체가 저장된 경우 best_estimator_ 추출
if hasattr(model_obj, 'best_estimator_'):
    model = model_obj.best_estimator_
else:
    model = model_obj


# =====================================================================
# 4. 단일 예측 및 성능 지표 계산
# =====================================================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Confusion Matrix 지표 계산
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 평가 지표 산출
accuracy = accuracy_score(y_test, y_pred)
fn_fp = fn + fp
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
recall = recall_score(y_test, y_pred, zero_division=0)
precision = precision_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
pr_auc = average_precision_score(y_test, y_prob)
auroc = roc_auc_score(y_test, y_prob)

# =====================================================================
# 5. 최종 결과 표 출력
# =====================================================================
results_tf = pd.DataFrame([{
    'Model': 'XGBoost (Tuned & Loaded)',
    'Accuracy (정확도)': round(accuracy, 4),
    'FN+FP (오분류 수)': fn_fp,
    'FPR (오탐률)': round(fpr, 4),
    'Recall (재현율)': round(recall, 4),
    'Precision (정밀도)': round(precision, 4),
    'F1-Score': round(f1, 4),
    'PR_AUC': round(pr_auc, 4),
    'AUROC': round(auroc, 4)
}])

print("[전처리 후 로드된 XGBoost 최적 모델 예측 성능 지표]")
print(results_tf.to_string(index=False))

[전처리 후 로드된 XGBoost 최적 모델 예측 성능 지표]
                   Model  Accuracy (정확도)  FN+FP (오분류 수)  FPR (오탐률)  Recall (재현율)  Precision (정밀도)  F1-Score  PR_AUC  AUROC
XGBoost (Tuned & Loaded)          0.9378             26     0.0263         0.875             0.95     0.911   0.971 0.9853


In [80]:
results_tf

,Model,Accuracy (정확도),FN+FP (오분류 수),FPR (오탐률),Recall (재현율),Precision (정밀도),F1-Score,PR_AUC,AUROC
0,XGBoost (Tuned & Loaded),0.9378,26,0.0263,0.875,0.95,0.911,0.971,0.9853


In [81]:
tuned_result

,Model,FN+FP (오분류 수),FPR (오탐률),Recall (재현율),Precision (정밀도),F1-Score,PR_AUC,AUROC
0,XGBoost (Tuned),27.4,0.102,0.7631,0.8235,0.792,0.8612,0.885


In [83]:
results_df.tail(1)

,Model,FN+FP (오분류 수),FPR (오탐률),Recall (재현율),Precision (정밀도),F1-Score,PR_AUC,AUROC
5,XGBoost,31.8,0.133,0.7484,0.7783,0.763,0.8484,0.8743
